# 06 — ML de regresión: ¿cuánto amplifica el suelo de cada sitio?
El modelo predice la amplificación del suelo `ln_amp` (ln(Sa·R) menos el término del sismo) a partir del periodo, el Vs30 y el perfil de suelo. Su champion arma la tabla F(T, Vs30) que usa `07_demanda_sismica` para calcular la demanda en cada edificio.
1. **Monitoreo:** cuando llega un sismo nuevo se mide el RMSE del champion en sus estaciones y el drift (PSI) de sus variables.
2. **Decisión de recalibrado:** si no hay evidencia, el notebook termina y se conserva el champion.
3. **Entrenamiento:** ventana expansiva con todos los sismos; el 30 % de las estaciones del sismo más reciente es el *holdout* y el 70 % restante pesa 3 veces. Se comparan 4 algoritmos de MLlib.
4. **Promoción:** el challenger reemplaza al champion si baja el RMSE al menos 5 %.
5. **MLflow:** la comparación queda registrada como experimento.

## 1. Setup — imports

In [ ]:
import json

import mlflow
import numpy as np
import pandas as pd

from pyspark.sql.functions import col, lit, when

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import (
    LinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor,
)
from pyspark.ml.evaluation import RegressionEvaluator

## 2. Catálogo, parámetros y umbrales

In [ ]:
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

dbutils.widgets.text("forzar_recalibrado", "false")
forzar = dbutils.widgets.get("forzar_recalibrado").lower() == "true"

In [ ]:
caso = "regresion"
target = "ln_amp"
SEED = 700

UMBRALES = {
    "psi_feature": 0.20,
    "n_features_drift": 2,
    "aumento_rmse": 0.40,
    "mejora_minima_rmse": 0.05,
}

## 3. Función de drift: PSI
Population Stability Index entre la distribución de referencia (datos del champion) y la del lote nuevo. PSI > 0.20 indica un cambio importante.

In [ ]:
def psi(referencia, actual, bins=10):
    ref = pd.Series(referencia).dropna().astype(float)
    act = pd.Series(actual).dropna().astype(float)
    cortes = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return 0.0
    cortes[0], cortes[-1] = -np.inf, np.inf
    p = np.histogram(ref, cortes)[0] / len(ref)
    q = np.histogram(act, cortes)[0] / len(act)
    p, q = np.clip(p, 1e-4, None), np.clip(q, 1e-4, None)
    return float(np.sum((q - p) * np.log(q / p)))

## 4. Datos y variables seleccionadas

In [ ]:
df = spark.table("fe_regresion")

seleccion = spark.table("fe_columnas_seleccionadas").filter(col("caso") == caso)
cols_nums = [r["columna"] for r in seleccion.filter(col("tipo") == "num").collect()]
cols_cat = [r["columna"] for r in seleccion.filter(col("tipo") == "cat").collect()]

lotes = sorted(r[0] for r in df.select("evento").distinct().collect())
lote_actual = lotes[-1]

print("Sismos:", lotes, "| sismo más reciente:", lote_actual)
print("Variables seleccionadas:", cols_nums, cols_cat)

## 5. Champion actual (registro de modelos)

In [ ]:
registro = spark.table("ml_registro_modelos").filter(col("caso") == caso)
fila_champion = registro.filter(col("promovido")).orderBy(col("version").desc()).first()

if fila_champion is None:
    champion, meta = None, None
else:
    champion = PipelineModel.load(fila_champion["ruta"])
    meta = {"version": fila_champion["version"],
            "cols_num": json.loads(fila_champion["columnas_num"]),
            "cols_cat": json.loads(fila_champion["columnas_cat"]),
            "snapshots": json.loads(fila_champion["snapshots"]),
            "metricas": json.loads(fila_champion["metricas"]),
            "detalle": json.loads(fila_champion["detalle"])}

print("Champion actual:", "v" + str(meta["version"]) if meta else "no existe")

## 6. Monitoreo: degradación de la métrica y drift

In [ ]:
n_drift, metrica_base, metrica_nueva, degradacion = 0, None, None, False
evaluador = RegressionEvaluator(labelCol=target, predictionCol="prediction")

if champion is not None and lote_actual not in meta["snapshots"]:
    ref = df.filter(col("evento").isin(meta["snapshots"])).select(meta["cols_num"]).toPandas()
    act = df.filter(col("evento") == lote_actual).select(meta["cols_num"]).toPandas()
    psis = {c: psi(ref[c], act[c]) for c in meta["cols_num"]}
    n_drift = sum(v > UMBRALES["psi_feature"] for v in psis.values())
    print("PSI:", [(c, round(v, 3)) for c, v in sorted(psis.items(), key=lambda kv: -kv[1])])

    metrica_base = meta["metricas"]["RMSE"]
    metrica_nueva = evaluador.setMetricName("rmse").evaluate(champion.transform(df.filter(col("evento") == lote_actual)))
    degradacion = metrica_nueva > metrica_base * (1 + UMBRALES["aumento_rmse"])
    print(f"RMSE base: {metrica_base:.3f} | RMSE en {lote_actual}: {metrica_nueva:.3f} | degradacion: {degradacion}")
elif champion is not None:
    print(f"El champion ya se entrenó con {lote_actual}: no hay sismo nuevo que monitorear")

## 7. Decisión de recalibrado
Se recalibra si no hay champion, si la métrica se degradó, si hay drift en 2 o más variables o si el job lo fuerza. La decisión queda en `ml_monitoreo`.

In [ ]:
if champion is None:
    reentrenar, motivo = True, "No existe champion: primer entrenamiento"
else:
    drift = n_drift >= UMBRALES["n_features_drift"]
    reentrenar = bool(degradacion or drift or forzar)
    senales = [("degradacion de la metrica", degradacion), (f"drift en {n_drift} variables", drift), ("forzado por parametro", forzar)]
    motivos = [texto for texto, activa in senales if activa]
    motivo = " + ".join(motivos) if motivos else "Sin cambios relevantes: se conserva el champion"

print("DECISION:", "RECALIBRAR" if reentrenar else "SIN CAMBIOS", "->", motivo)

In [ ]:
clave = f"{lote_actual}|{caso}"
fila = [(clave, lote_actual, caso, meta["version"] if meta else None, int(n_drift), metrica_base, metrica_nueva,
         bool(degradacion), bool(reentrenar), motivo)]
esquema_monitoreo = ("clave string, lote string, caso string, champion_version int, n_features_drift int, "
                     "metrica_base double, metrica_nueva double, degradacion boolean, reentrenar boolean, motivo string")
monitoreo = spark.createDataFrame(fila, esquema_monitoreo)

monitoreo.write.mode("overwrite").option("replaceWhere", f"caso = '{caso}' AND lote = '{lote_actual}'").saveAsTable("ml_monitoreo")

In [ ]:
if not reentrenar:
    dbutils.notebook.exit("Sin cambios: se conserva el champion")

## 8. Entrenamiento

In [ ]:
holdout = df.filter(col("es_holdout"))
train = (df.filter(~col("es_holdout"))
         .withColumn("peso", when(col("evento") == lote_actual, lit(3.0)).otherwise(lit(1.0))))
snapshots_train = sorted(r[0] for r in train.select("evento").distinct().collect())

print("train:", train.count(), "| holdout:", holdout.count(), "filas")
print("estaciones en el holdout:", holdout.select("estacion").distinct().count())
print("Sismos de entrenamiento:", snapshots_train)

In [ ]:
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cols_cat]
encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in cols_cat], outputCols=[f"{c}_vec" for c in cols_cat], handleInvalid="keep")
num_assembler = VectorAssembler(inputCols=cols_nums, outputCol="num_features")
scaler = StandardScaler(inputCol="num_features", outputCol="num_scaled", withMean=True, withStd=True)
final_assembler = VectorAssembler(inputCols=["num_scaled"] + [f"{c}_vec" for c in cols_cat], outputCol="features_scaled")

preparacion = (indexers + [encoder] if cols_cat else []) + [num_assembler, scaler, final_assembler]

In [ ]:
comunes = dict(labelCol=target, featuresCol="features_scaled", weightCol="peso")

algoritmos = {
    "LinearRegression": LinearRegression(**comunes),
    "DecisionTree": DecisionTreeRegressor(maxDepth=8, minInstancesPerNode=50, seed=SEED, **comunes),
    "RandomForest": RandomForestRegressor(numTrees=100, maxDepth=8, minInstancesPerNode=50, seed=SEED, **comunes),
    "GBT": GBTRegressor(maxIter=50, maxDepth=4, seed=SEED, **comunes),
}

In [ ]:
modelos, predicciones = {}, {}

for nombre, estimador in algoritmos.items():
    pipeline = Pipeline(stages=preparacion + [estimador])
    modelos[nombre] = pipeline.fit(train)
    predicciones[nombre] = modelos[nombre].transform(holdout)
    print("entrenado:", nombre)

if champion is not None:
    predicciones["champion"] = champion.transform(holdout)

## 9. Evaluación en el holdout

In [ ]:
filas = []
for nombre, pred in predicciones.items():
    filas.append({
        "Modelo": nombre,
        "RMSE": evaluador.setMetricName("rmse").evaluate(pred),
        "MAE": evaluador.setMetricName("mae").evaluate(pred),
        "R2": evaluador.setMetricName("r2").evaluate(pred),
    })

tabla = pd.DataFrame(filas).sort_values("RMSE").reset_index(drop=True)
display(tabla.round(4))

## 10. Promoción champion / challenger

In [ ]:
challengers = tabla[tabla["Modelo"] != "champion"].reset_index(drop=True)
ganador = challengers.loc[0, "Modelo"]
rmse_ganador = float(challengers.loc[0, "RMSE"])

if champion is None:
    promover = True
else:
    rmse_champion = float(tabla.loc[tabla["Modelo"] == "champion", "RMSE"].iloc[0])
    promover = rmse_ganador <= rmse_champion * (1 - UMBRALES["mejora_minima_rmse"])

print("Mejor algoritmo:", ganador, "| RMSE:", round(rmse_ganador, 4))
print("Promover challenger a champion:", promover)

## 11. Interpretación del mejor algoritmo

In [ ]:
etapa = modelos[ganador].stages[-1]
nombres = cols_nums + [f"{c}={v}" for c, ind in zip(cols_cat, [s for s in modelos[ganador].stages if hasattr(s, "labels")]) for v in ind.labels]
if hasattr(etapa, "featureImportances"):
    pesos = etapa.featureImportances.toArray()
else:
    pesos = etapa.coefficients.toArray()
nombres = (nombres + [f"variable_{i}" for i in range(len(pesos))])[:len(pesos)]
importancia = pd.Series(pesos, index=nombres).sort_values(key=abs, ascending=False)
display(importancia.round(4).to_frame("importancia_o_coeficiente"))

## 12. Guardar el modelo en el Volume y registrarlo

In [ ]:
metricas = challengers.loc[0, ["RMSE", "MAE", "R2"]].astype(float).to_dict()
detalle = {"comparacion": challengers.round(4).to_dict("records"), "importancia": importancia.round(4).to_dict()}

In [ ]:
version = registro.count() + 1
ruta = f"{volumen}/modelos/{caso}/v{version}"
modelos[ganador].write().overwrite().save(ruta)

fila = [(caso, version, ganador, lote_actual, ruta, bool(promover), json.dumps(cols_nums), json.dumps(cols_cat),
         json.dumps(snapshots_train), json.dumps(metricas), json.dumps(detalle, default=float))]
esquema_registro = ("caso string, version int, algoritmo string, lote string, ruta string, promovido boolean, "
                    "columnas_num string, columnas_cat string, snapshots string, metricas string, detalle string")
spark.createDataFrame(fila, esquema_registro).write.mode("append").saveAsTable("ml_registro_modelos")

print(f"Version v{version} guardada en {ruta}")
print("Estado:", "CHAMPION" if promover else "challenger (se conserva el champion actual)")

## 13. MLflow — experimento
Cada ejecución del notebook queda registrada en un experimento de MLflow (un *run* padre por ejecución y un *run* hijo por algoritmo) para comparar parámetros y métricas desde la interfaz.

In [ ]:
usuario = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/sismos_{caso}")

In [ ]:
with mlflow.start_run(run_name=f"{caso}_{lote_actual}"):
    mlflow.log_params({"sismo": lote_actual, "forzado": forzar, "n_train": train.count(), "n_holdout": holdout.count(),
                       "snapshots_train": ",".join(snapshots_train), "variables": ",".join(cols_nums + cols_cat),
                       "ganador": ganador, "promovido": bool(promover), "version_registro": version, "ruta_modelo": ruta})
    mlflow.log_metric("rmse_ganador", rmse_ganador)
    if champion is not None:
        mlflow.log_metric("rmse_champion", rmse_champion)
    mlflow.log_text(challengers.to_csv(index=False), "comparacion.csv")

    for _, fila_modelo in challengers.iterrows():
        with mlflow.start_run(run_name=fila_modelo["Modelo"], nested=True):
            mlflow.log_param("algoritmo", fila_modelo["Modelo"])
            mlflow.log_metrics({"rmse": float(fila_modelo["RMSE"]), "mae": float(fila_modelo["MAE"]), "r2": float(fila_modelo["R2"])})